# Importing the neccessary Libraries

In [1]:
import os
import uuid
import base64
from dotenv import load_dotenv

#libraries for the scanned textbooks
import fitz
import pytesseract
from PIL import Image

#Langchain libraries
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_pinecone import PineconeVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_groq import ChatGroq

c:\Users\Loba\MB_ASSISTANT\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
c:\Users\Loba\MB_ASSISTANT\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.documents import Document
from langchain_community.document_loaders import DirectoryLoader

In [4]:
import os
load_dotenv()

True

Let us load the scanned copies
-----------------------------------------------
PyMuPixmap to PIL
1. fitz opens the PDF and takes a "screenshot" (pixmap) of the page.
2. We convert that raw screenshot into a standard PIL Image.
3. We hand the PIL Image to Tesseract to read the words.
-----------------------------------------------------------

In [5]:
#Instantiating pytesseract
pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'

In [ ]:
# Define function to load scanned textbooks
def extract_scanned_textbooks(pdf_path):
    """
    this function uses PyMuPDf (fitz) and Tesseract to convert scanned pdfs to texts
    """
    print(f"Opening {pdf_path} with PyMuPDF...")
    doc = fitz.open(pdf_path)
    docs = []

    for page_num, page in enumerate(doc):
        if page_num % 200 == 0:
            print(f"Processed {page_num} pages...")

        # Take High Resolution Screenshot of the page (matrix scales it up for better OCR reading)
        pix = page.get_pixmap(matrix=fitz.Matrix(2, 2))
        
        #Convert to PIL image for Tesseract
        img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)

        #Extract text using Tesseract
        extracted_text = pytesseract.image_to_string(img)

        #keeping only pages that have text
        if extracted_text.strip():
            docs.append(
                Document(
                    page_content=extracted_text,
                    #Include source
                    metadata={
                        "source": pdf_path, 
                        "page": page_num + 1, 
                        "type": "ocr_text"
                        }
                )
            )
    
    return docs

In [7]:
# let's define a function to load all the pdfs from our scanned textbooks directory
def load_scanned_directory(directory_path):
    """
    Acts as a pdf Directory loader for the scanned OCR pdfs.
    loops through the folder and processes every pdf it finds.
    """
    all_scanned_docs = []

    for filename in os.listdir(directory_path):
        #combine folder path and file name
        full_file_path = os.path.join(directory_path, filename)
        print(f"\n Starting OCR on Textbook: {filename}")

        #use the PyMuPDF function
        book_docs = extract_scanned_textbooks(full_file_path)

        #add this book's pages  to out master list
        all_scanned_docs.extend(book_docs)
    
    return all_scanned_docs

In [14]:
#This should work, but it will take a lot of time, so i might just try using paddle OCR on colab and use T4 GPU
# Let's load the scanned textbooks
folder = r"data\scanned_textbooks"

# Check if the variable exists in memory AND is not empty
if 'total_scanned_docs' in locals() and total_scanned_docs:
    print(f"Skipping OCR... {len(total_scanned_docs)} pages already exist in memory!")
else:
    print("Starting Batch OCR Process....")
    total_scanned_docs = load_scanned_directory(folder)
    print(f"\n Batch Complete! Extracted a total of {len(total_scanned_docs)} pages across all textbooks.")

Skipping OCR... 1319 pages already exist in memory!


In [10]:
# Define exactly where you want the final text file saved
output_filepath = r"data\scanned_textbooks\extracted_board_prep.txt"

print(f"Writing {len(total_scanned_docs)} pages to local text file...")

# Open the file in write mode ('w') with utf-8 encoding to prevent character crashes
with open(output_filepath, "w", encoding="utf-8") as f:
    
    for doc in total_scanned_docs:
        # 1. Safely extract the tracking data from the metadata dictionary
        source_name = doc.metadata.get("source", "Unknown_Source")
        page_num = doc.metadata.get("page", "Unknown")
        
        # 2. Write a clear header so LangChain can chunk it easily later
        f.write(f"\n\n--- SOURCE: {source_name} | PAGE: {page_num} ---\n\n")
        
        # 3. Write the actual medical text extracted by Tesseract
        f.write(doc.page_content)

print(f"✅ Success! All OCR text saved locally to: {output_filepath}")

Writing 1319 pages to local text file...
✅ Success! All OCR text saved locally to: data\scanned_textbooks\extracted_board_prep.txt
